# 07 - Gold Layer

## Mục tiêu
- Đọc dữ liệu đã được làm sạch từ Silver Layer
- Join các bảng nghiệp vụ
- Tạo các bảng Gold phục vụ phân tích và dashboard
- Tổng hợp doanh thu, sản phẩm, khách hàng, review và thanh toán
- Ghi các bảng Gold dưới dạng Apache Iceberg


## 7.1. Tạo SparkSession


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ICEBERG_VERSION = "1.11.0"
HADOOP_VERSION = "3.4.2"

PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:{ICEBERG_VERSION},"
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
    f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}"
)

AWS_ACCESS_KEY_ID = (
    os.environ.get("AWS_ACCESS_KEY_ID")
    or os.environ.get("MINIO_ROOT_USER")
)

AWS_SECRET_ACCESS_KEY = (
    os.environ.get("AWS_SECRET_ACCESS_KEY")
    or os.environ.get("MINIO_ROOT_PASSWORD")
)

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise RuntimeError("Không tìm thấy credential MinIO trong biến môi trường.")

spark = (
    SparkSession.builder
    .appName("EcommerceGoldLayer")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://iceberg-rest:8181"
    )
    .config(
        "spark.sql.catalog.lakehouse.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

print("✅ SparkSession created")
print("Spark version:", spark.version)


## 7.2. Kiểm tra Silver Layer


In [ ]:
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)


## 7.3. Đọc các bảng Silver


In [ ]:
customers = spark.table("lakehouse.silver.customers")
products = spark.table("lakehouse.silver.products")
orders = spark.table("lakehouse.silver.orders")
order_items = spark.table("lakehouse.silver.order_items")
payments = spark.table("lakehouse.silver.payments")
reviews = spark.table("lakehouse.silver.reviews")

silver_tables = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews
}

for name, df in silver_tables.items():
    print(f"{name:15} {df.count():,} rows")


## 7.4. Tạo `sales_detail`

Join:
- `order_items` + `orders` qua `order_id`
- `products` qua `product_id`
- `customers` qua `customer_id`

Không join trực tiếp `payments` vì một order có thể có nhiều item và nhiều payment, dễ làm nhân bản dòng.


In [ ]:
sales_detail = (
    order_items.alias("oi")
    .join(
        orders.alias("o"),
        F.col("oi.order_id") == F.col("o.order_id"),
        "inner"
    )
    .join(
        products.alias("p"),
        F.col("oi.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        customers.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("oi.order_item_id").alias("order_item_id"),
        F.col("oi.product_id").alias("product_id"),
        F.col("p.product_category_name").alias("product_category_name"),
        F.col("o.order_status").alias("order_status"),
        F.col("o.order_purchase_timestamp").alias("order_purchase_timestamp"),
        F.col("c.customer_city").alias("customer_city"),
        F.col("c.customer_state").alias("customer_state"),
        F.col("oi.price").alias("price"),
        F.col("oi.freight_value").alias("freight_value")
    )
    .withColumn("order_date", F.to_date(F.col("order_purchase_timestamp")))
    .withColumn("item_total", F.col("price") + F.col("freight_value"))
    .withColumn("_generated_at", F.current_timestamp())
)

sales_detail.show(10, truncate=False)


## 7.5. Kiểm tra `sales_detail`


In [ ]:
print("Rows:", sales_detail.count())
print("NULL order_id:", sales_detail.filter(F.col("order_id").isNull()).count())
print("Negative item_total:", sales_detail.filter(F.col("item_total") < 0).count())


## 7.6. Ghi `gold.sales_detail`


In [ ]:
(
    sales_detail
    .writeTo("lakehouse.gold.sales_detail")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.sales_detail")


## 7.7. Tạo `daily_sales`


In [ ]:
daily_sales = (
    sales_detail
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items"),
        F.sum("price").alias("product_revenue"),
        F.sum("freight_value").alias("freight_revenue"),
        F.sum("item_total").alias("total_revenue"),
        F.avg("item_total").alias("avg_item_value")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

daily_sales.orderBy("order_date").show(20, truncate=False)

(
    daily_sales
    .writeTo("lakehouse.gold.daily_sales")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.daily_sales")


## 7.8. Tạo `category_sales`


In [ ]:
category_sales = (
    sales_detail
    .groupBy("product_category_name")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("items_sold"),
        F.sum("price").alias("product_revenue"),
        F.sum("freight_value").alias("freight_revenue"),
        F.sum("item_total").alias("total_revenue")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

category_sales.orderBy(F.col("total_revenue").desc()).show(20, truncate=False)

(
    category_sales
    .writeTo("lakehouse.gold.category_sales")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.category_sales")


## 7.9. Tạo `product_performance`


In [ ]:
product_performance = (
    sales_detail
    .groupBy("product_id", "product_category_name")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("items_sold"),
        F.sum("price").alias("product_revenue"),
        F.avg("price").alias("avg_price"),
        F.sum("freight_value").alias("total_freight"),
        F.sum("item_total").alias("total_revenue")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

product_performance.orderBy(F.col("product_revenue").desc()).show(20, truncate=False)

(
    product_performance
    .writeTo("lakehouse.gold.product_performance")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.product_performance")


## 7.10. Tạo `customer_summary`


In [ ]:
customer_summary = (
    sales_detail
    .groupBy("customer_id", "customer_city", "customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items"),
        F.sum("item_total").alias("total_spent"),
        F.avg("item_total").alias("avg_item_value"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

customer_summary.orderBy(F.col("total_spent").desc()).show(20, truncate=False)

(
    customer_summary
    .writeTo("lakehouse.gold.customer_summary")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.customer_summary")


## 7.11. Tạo `review_summary`


In [ ]:
total_reviews = reviews.count()

review_summary = (
    reviews
    .groupBy("review_score")
    .agg(F.count("*").alias("total_reviews"))
    .withColumn(
        "percentage",
        F.round(F.col("total_reviews") / F.lit(total_reviews) * 100, 2)
    )
    .withColumn("_generated_at", F.current_timestamp())
)

review_summary.orderBy("review_score").show(truncate=False)

(
    review_summary
    .writeTo("lakehouse.gold.review_summary")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.review_summary")


## 7.12. Tạo `state_sales`


In [ ]:
state_sales = (
    sales_detail
    .groupBy("customer_state")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items"),
        F.sum("item_total").alias("total_revenue")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

state_sales.orderBy(F.col("total_revenue").desc()).show(truncate=False)

(
    state_sales
    .writeTo("lakehouse.gold.state_sales")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.state_sales")


## 7.13. Tạo `payment_summary`

Aggregate payments theo `order_id` trước để tránh nhân bản dữ liệu khi join với order items.


In [ ]:
payment_summary = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_total"),
        F.count("*").alias("payment_count"),
        F.max("payment_installments").alias("max_installments")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

payment_summary.show(10, truncate=False)

(
    payment_summary
    .writeTo("lakehouse.gold.payment_summary")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.payment_summary")


## 7.14. Tạo `order_financial_summary`


In [ ]:
sales_by_order = (
    sales_detail
    .groupBy("order_id")
    .agg(F.sum("item_total").alias("sales_total"))
)

order_financial_summary = (
    sales_by_order.alias("s")
    .join(
        payment_summary.alias("p"),
        F.col("s.order_id") == F.col("p.order_id"),
        "left"
    )
    .select(
        F.col("s.order_id").alias("order_id"),
        F.col("s.sales_total").alias("sales_total"),
        F.col("p.payment_total").alias("payment_total"),
        F.col("p.payment_count").alias("payment_count"),
        F.col("p.max_installments").alias("max_installments")
    )
    .withColumn(
        "payment_difference",
        F.col("payment_total") - F.col("sales_total")
    )
    .withColumn("_generated_at", F.current_timestamp())
)

order_financial_summary.show(10, truncate=False)

(
    order_financial_summary
    .writeTo("lakehouse.gold.order_financial_summary")
    .using("iceberg")
    .createOrReplace()
)
print("✅ gold.order_financial_summary")


## 7.15. Kiểm tra Gold tables


In [ ]:
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)


## 7.16. Kiểm tra số dòng


In [ ]:
GOLD_TABLES = [
    "sales_detail",
    "daily_sales",
    "category_sales",
    "product_performance",
    "customer_summary",
    "review_summary",
    "state_sales",
    "payment_summary",
    "order_financial_summary"
]

for table in GOLD_TABLES:
    count = spark.table(f"lakehouse.gold.{table}").count()
    print(f"{table:25} {count:,} rows")


## 7.17. Data Quality Gold


In [ ]:
gold_sales = spark.table("lakehouse.gold.sales_detail")

print("NULL order_id:", gold_sales.filter(F.col("order_id").isNull()).count())
print("NULL item_total:", gold_sales.filter(F.col("item_total").isNull()).count())
print("Negative item_total:", gold_sales.filter(F.col("item_total") < 0).count())


## 7.18. KPI tổng doanh thu


In [ ]:
spark.sql("""
SELECT
    SUM(price) AS product_revenue,
    SUM(freight_value) AS freight_revenue,
    SUM(item_total) AS total_revenue
FROM lakehouse.gold.sales_detail
""").show(truncate=False)


## 7.19. Top 10 category


In [ ]:
spark.sql("""
SELECT
    product_category_name,
    total_orders,
    items_sold,
    total_revenue
FROM lakehouse.gold.category_sales
ORDER BY total_revenue DESC
LIMIT 10
""").show(truncate=False)


## 7.20. Top 10 sản phẩm


In [ ]:
spark.sql("""
SELECT
    product_id,
    product_category_name,
    items_sold,
    product_revenue
FROM lakehouse.gold.product_performance
ORDER BY product_revenue DESC
LIMIT 10
""").show(truncate=False)


## 7.21. Top khách hàng


In [ ]:
spark.sql("""
SELECT
    customer_id,
    customer_city,
    customer_state,
    total_orders,
    total_spent
FROM lakehouse.gold.customer_summary
ORDER BY total_spent DESC
LIMIT 10
""").show(truncate=False)


## 7.22. Kiểm tra Iceberg snapshot


In [ ]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.gold.daily_sales.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)


## 7.23. Checklist hoàn thành


In [ ]:
expected_gold_tables = {
    "sales_detail",
    "daily_sales",
    "category_sales",
    "product_performance",
    "customer_summary",
    "review_summary",
    "state_sales",
    "payment_summary",
    "order_financial_summary"
}

actual_gold_tables = {
    row.tableName
    for row in spark.sql("SHOW TABLES IN lakehouse.gold").collect()
}

print("========== GOLD LAYER ==========")

for table in sorted(expected_gold_tables):
    if table in actual_gold_tables:
        print(f"✅ {table}")
    else:
        print(f"❌ {table}")

print("✅ Step 07 hoàn tất")
